# Multimodal RAG Evaluation & Benchmarking Suite

This notebook implements a comprehensive **LLM Evaluation Framework** for the NovaCore Multimodal RAG pipeline. It measures:

1. **Modality Routing Accuracy**: Verifying whether the system accurately selects the Vision LLM (for charts and diagrams) vs. the Text LLM (for narrative text and tables).
2. **Retrieval Precision (Hit Rate)**: Ensuring the retriever fetches the correct source pages and modalities in top-$k$.
3. **Faithfulness / Groundedness (LLM-as-a-Judge)**: Detecting hallucinations by verifying that every generated claim is strictly grounded in the retrieved context.
4. **Answer Relevance & Correctness (LLM-as-a-Judge)**: Evaluating semantic alignment between generated responses and ground truth factual targets.
5. **Key Fact Coverage**: Deterministic validation of essential numerical and entity tokens.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure root src is in Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_pipeline import MultimodalRAGPipeline
from src.evaluator import MultimodalRAGEvaluator, BENCHMARK_DATASET

print("Evaluation modules successfully loaded!")

## 2. Inspect Golden Test Benchmark Dataset

The benchmark covers **Text**, **Table**, **Visual (Charts & Diagrams)**, and **Cross-Modal** test cases.

In [ ]:
benchmark_df = pd.DataFrame([
    {
        "ID": item["id"],
        "Category": item["category"],
        "Question": item["question"],
        "Expected Visual Routing": item["expected_visual_routing"],
        "Expected Pages": item["expected_pages"],
        "Key Facts Required": ", ".join(item["key_facts"])
    }
    for item in BENCHMARK_DATASET
])

benchmark_df

## 3. Run Multimodal RAG Pipeline Benchmark

Each query executes through Pinecone retrieval, dynamic routing (VLM vs Text LLM), and is scored by an impartial LLM Judge with strict rubrics.

In [ ]:
# Initialize pipeline & evaluator
pipeline = MultimodalRAGPipeline()
evaluator = MultimodalRAGEvaluator(pipeline=pipeline)

# Run the full benchmark
results_df = evaluator.run_benchmark(BENCHMARK_DATASET)

## 4. Evaluation Results Breakdown

Let's review the per-query scores, routing decisions, and pass status.

In [ ]:
display_cols = [
    "ID", "Category", "Routed To", "Routing Match", 
    "Retrieval Hit", "Key Fact Score", "Faithfulness (1-5)", 
    "Relevance (1-5)", "Overall Passed"
]
results_df[display_cols]

## 5. Aggregate Scorecard & Modality Breakdown

In [ ]:
summary = evaluator.generate_summary_report(results_df)

print("=" * 45)
print("     MULTIMODAL RAG BENCHMARK SCORECARD     ")
print("=" * 45)
for k, v in summary.items():
    if k != "Breakdown":
        print(f"{k:<32}: {v}")

print("\n--- MODALITY BREAKDOWN ---")
display(summary["Breakdown"])

## 6. Benchmark Visualizations

Plotting performance metrics across modalities.

In [ ]:
sns.set_theme(style="whitegrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Scores across categories
cat_scores = results_df.groupby("Category")[["Faithfulness (1-5)", "Relevance (1-5)"]].mean()
cat_scores.plot(kind="bar", ax=ax1, colormap="viridis", rot=0, ylim=(0, 5.5))
ax1.set_title("Mean LLM-as-a-Judge Scores by Modality", fontsize=13, fontweight="bold")
ax1.set_ylabel("Score (1-5 Scale)")
ax1.axhline(4.0, color="red", linestyle="--", label="Pass Threshold (4.0)")
ax1.legend()

# Plot 2: Routing & Fact Accuracy
cat_acc = results_df.groupby("Category")[["Routing Match", "Key Fact Score"]].mean() * 100
cat_acc.plot(kind="bar", ax=ax2, colormap="crest", rot=0, ylim=(0, 110))
ax2.set_title("Routing & Key Fact Accuracy by Modality", fontsize=13, fontweight="bold")
ax2.set_ylabel("Accuracy (%)")
ax2.legend(["Routing Accuracy", "Key Fact Retention"])

plt.tight_layout()
plt.show()

## 7. LLM-as-a-Judge Reasoning Inspector

Let's inspect the qualitative rationale provided by the evaluator for each response.

In [ ]:
for idx, row in results_df.iterrows():
    print(f"🔹 [{row['ID']}] {row['Question']}")
    print(f"   Routed: {row['Routed To']} (Expected Visual: {row['Expected Visual']})")
    print(f"   Faithfulness ({row['Faithfulness (1-5)']}/5): {row['Faithfulness Reason']}")
    print(f"   Relevance ({row['Relevance (1-5)']}/5): {row['Relevance Reason']}")
    print("-" * 80)